In [12]:
import numpy as np
import requests

import requests
from datetime import datetime, timedelta
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import johnsonsu
from scipy.optimize import minimize
import csv
import time

#1. Getting prices
def prices(startd,endd,tic):
    info=[]
    key='&apikey=ZKMMTO1ATDBLXH2K'
    ticker='&symbol='+str(tic)
#    endpoint='function=TIME_SERIES_MONTHLY_ADJUSTED'
    endpoint='function=TIME_SERIES_DAILY_ADJUSTED'
    size='&outputsize=full'
    web='https://www.alphavantage.co/query?'
    url =web+endpoint+ticker+size+key

    r = requests.get(url)
    if r.status_code==200:
        print('connection successful')
        data = r.json() #need to convert to json to navigate
        r1=data.get('Time Series (Daily)', {})
        #r1=data.get('Monthly Adjusted Time Series', {})
        r2=data['Time Series (Daily)']
        #r2=data['Monthly Adjusted Time Series']
        
        for date, values in sorted(r1.items()):
            if startd <= date <= endd:
                info.append([tic, date, values['5. adjusted close']])
    return info
    

    

def compute_log_returns(data, freq='daily'):
    """
    Computes log returns at specified frequency.
    
    Parameters:
    -----------
    data : list of lists
        Price data: [[ticker, date, adjusted_close], ...]
    freq : str
        'daily', 'weekly', 'monthly', or 'yearly'
    
    Returns:
    --------
    list of lists: [[ticker, date, adjusted_close, log_return], ...]
    """
    sorted_data = sorted(data, key=lambda x: x[1])  # Sort by date (index 1)
    data_with_logret = []
    
    if freq == 'daily':
        for i in range(len(sorted_data)):
            if i == 0:
                data_with_logret.append(sorted_data[i] + [None])
            else:
                # Convert to float before taking log
                price_current = float(sorted_data[i][2])
                price_previous = float(sorted_data[i-1][2])
                log_return = np.log(price_current) - np.log(price_previous)
                data_with_logret.append(sorted_data[i] + [log_return])
    
    elif freq == 'weekly':
        if len(sorted_data) == 0:
            return data_with_logret
        data_with_logret.append(sorted_data[0] + [None])
        last_kept_idx = 0
        for i in range(1, len(sorted_data)):
            d30 = days_30_360(sorted_data[last_kept_idx][1], sorted_data[i][1])
            if d30 >= 7:
                # Convert to float before taking log
                price_current = float(sorted_data[i][2])
                price_previous = float(sorted_data[last_kept_idx][2])
                log_return = np.log(price_current) - np.log(price_previous)
                data_with_logret.append(sorted_data[i] + [log_return])
                last_kept_idx = i
    
    elif freq == 'monthly':
        if len(sorted_data) == 0:
            return data_with_logret
        data_with_logret.append(sorted_data[0] + [None])
        last_kept_idx = 0
        for i in range(1, len(sorted_data)):
            d30 = days_30_360(sorted_data[last_kept_idx][1], sorted_data[i][1])
            if d30 >= 30:
                # Convert to float before taking log
                price_current = float(sorted_data[i][2])
                price_previous = float(sorted_data[last_kept_idx][2])
                log_return = np.log(price_current) - np.log(price_previous)
                data_with_logret.append(sorted_data[i] + [log_return])
                last_kept_idx = i
    
    elif freq == 'yearly':
        if len(sorted_data) == 0:
            return data_with_logret
        data_with_logret.append(sorted_data[0] + [None])
        last_kept_idx = 0
        for i in range(1, len(sorted_data)):
            d30 = days_30_360(sorted_data[last_kept_idx][1], sorted_data[i][1])
            if d30 >= 360:
                # Convert to float before taking log
                price_current = float(sorted_data[i][2])
                price_previous = float(sorted_data[last_kept_idx][2])
                log_return = np.log(price_current) - np.log(price_previous)
                data_with_logret.append(sorted_data[i] + [log_return])
                last_kept_idx = i
    
    return data_with_logret


#6. Download, compute returns, and export to CSV
def download_and_export(ticker, startd, endd, freq='daily', path=None):
    """
    Downloads price data, computes log and simple returns, and exports to CSV.

    Parameters:
    -----------
    ticker : str
        Stock ticker symbol (e.g. 'NVDA')
    startd : str
        Start date 'YYYY-MM-DD'
    endd : str
        End date 'YYYY-MM-DD'
    freq : str
        'daily', 'weekly', 'monthly', or 'yearly'
    path : str
        File path for CSV output. Defaults to '{ticker}_returns.csv'

    Returns:
    --------
    list of lists: [[ticker, date, price, log_return, simple_return], ...]
    First row is a header row.
    """
    if path is None:
        path = f"{ticker}_returns.csv"

    raw = prices(startd, endd, ticker)
    with_log = compute_log_returns(raw, freq=freq)
    with_both = compute_returns(with_log)

    with open(path, mode='w', newline='') as f:
        writer = csv.writer(f)
        for row in with_both:
            writer.writerow(row)

    print(f"Exported {len(with_both)-1} rows to {path}")
    return with_both


In [13]:
data_nvda = prices("2022-01-01", "2025-12-31", "NVDA")
time.sleep(5)
data_jpm = prices("2022-01-01", "2025-12-31", "JPM")
time.sleep(5)
data_aapl = prices("2022-01-01", "2025-12-31", "AAPL")
time.sleep(5)
data_spy = prices("2022-01-01", "2025-12-31", "SPY")

connection successful
connection successful
connection successful


In [14]:
daily_nvda = compute_log_returns(data_nvda, freq='daily')
daily_jpm = compute_log_returns(data_jpm, freq='daily')
daily_aapl = compute_log_returns(data_aapl, freq='daily')
daily_spy = compute_log_returns(data_spy, freq='daily')

In [ ]:
def compute_returns(data):
    """
    Computes simple and log returns from compute_log_returns output.
    
    Parameters:
    -----------
    data : list of lists
        [[ticker, date, adjusted_close, log_return], ...]
    
    Returns:
    --------
    list of lists: [[ticker, date, adjusted_close, log_return, simple_return], ...]
    First row is a header row.
    """
    header = ["ticker", "date", "price", "log_return", "simple_return"]
    result = [header]
    for i in range(len(data)):
        if i == 0 or data[i-1][2] is None:
            result.append(data[i][:4] + [None])
        else:
            price_current = float(data[i][2])
            price_previous = float(data[i-1][2])
            simple_ret = (price_current - price_previous) / price_previous
            result.append(data[i][:4] + [simple_ret])
    return result


In [15]:
nvda = compute_returns(daily_nvda)
jpm = compute_returns(daily_jpm)
aapl = compute_returns(daily_aapl)
spy = compute_returns(daily_spy)

spy

[['ticker', 'date', 'price', 'log_return', 'simple_return'],
 ['SPY', '2022-01-03', '450.577268265127', None, None],
 ['SPY',
  '2022-01-04',
  '450.42635586446',
  np.float64(-0.0003349873364255629),
  -0.00033493123443187746],
 ['SPY',
  '2022-01-05',
  '441.777188901258',
  np.float64(-0.019388934217042575),
  -0.01920217778243122],
 ['SPY',
  '2022-01-06',
  '441.362179799425',
  np.float64(-0.0009398496932417899),
  -0.0009394081728510255],
 ['SPY',
  '2022-01-07',
  '439.617255166718',
  np.float64(-0.0039613340453916734),
  -0.003953498311749286],
 ['SPY',
  '2022-01-10',
  '439.070197714302',
  np.float64(-0.0012451697615674107),
  -0.0012443948593612577],
 ['SPY',
  '2022-01-11',
  '443.069376331966',
  np.float64(0.009067059530073251),
  0.00910828983265722],
 ['SPY',
  '2022-01-12',
  '444.267243512256',
  np.float64(0.002699917666330265),
  0.0027035657264485057],
 ['SPY',
  '2022-01-13',
  '438.145859260219',
  np.float64(-0.013874414215683473),
  -0.013778608127043061],
 

In [18]:
import os
os.makedirs("Group_Proj_CSVs", exist_ok=True)

nvda = download_and_export("NVDA", "2022-01-01", "2025-12-31", freq="daily", path="Group_Proj_CSVs/nvda_returns.csv")
time.sleep(5)
jpm  = download_and_export("JPM",  "2022-01-01", "2025-12-31", freq="daily", path="Group_Proj_CSVs/jpm_returns.csv")
time.sleep(5)
aapl = download_and_export("AAPL", "2022-01-01", "2025-12-31", freq="daily", path="Group_Proj_CSVs/aapl_returns.csv")
time.sleep(5)
spy  = download_and_export("SPY",  "2022-01-01", "2025-12-31", freq="daily", path="Group_Proj_CSVs/spy_returns.csv")

connection successful
Exported 1003 rows to Group_Proj_CSVs/nvda_returns.csv
connection successful
Exported 1003 rows to Group_Proj_CSVs/jpm_returns.csv
connection successful
Exported 1003 rows to Group_Proj_CSVs/aapl_returns.csv
connection successful
Exported 1003 rows to Group_Proj_CSVs/spy_returns.csv
